# Short-term wildfire risk forecast

This notebook calculates short-term wildfire risk for Deliblato Sands using:

- static fire susceptibility raster
- ECMWF near-real-time IFS forecast
- protected-area averaged meteorological conditions
- dynamic fire-weather modifier

## 1. Install packages if needed

Run this in Anaconda Prompt if the packages are missing:

```bash
conda install -c conda-forge earthengine-api geemap
```

In [1]:
import ee
import geemap

## 2. Authenticate and initialize Earth Engine

In [2]:
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    # CHANGE PROJECT NAME
    ee.Initialize(project='ee-minucher')

## 3. Load assets

In [3]:
roi = ee.FeatureCollection("projects/ee-minucher/assets/DeliblatoROI_SQFIN")
roisrp = ee.FeatureCollection("projects/ee-minucher/assets/SRPDeliblatoWGS84")
stat4 = ee.Image("projects/ee-minucher/assets/StaticV4")

roi_geom = roi.geometry()
roisrp_geom = roisrp.geometry()

## 4. Load ECMWF forecast collection

In [4]:
fc = ee.ImageCollection("ECMWF/NRT_FORECAST/IFS/OPER")

latest_creation = ee.Number(fc.aggregate_max("creation_time"))
latest_run = fc.filter(ee.Filter.eq("creation_time", latest_creation))

print("Latest forecast run:", ee.Date(latest_creation).format("YYYY-MM-dd HH:mm").getInfo(), "UTC")

forecast_dates = (
    ee.List(latest_run.aggregate_array("forecast_time"))
    .map(lambda t: ee.Date(ee.Number(t)).format("YYYY-MM-dd"))
    .distinct()
    .sort()
    .getInfo()
)

print("Available forecast dates:")
forecast_dates

Latest forecast run: 2026-04-30 12:00 UTC
Available forecast dates:


['2026-04-30',
 '2026-05-01',
 '2026-05-02',
 '2026-05-03',
 '2026-05-04',
 '2026-05-05',
 '2026-05-06',
 '2026-05-07',
 '2026-05-08',
 '2026-05-09',
 '2026-05-10',
 '2026-05-11',
 '2026-05-12',
 '2026-05-13',
 '2026-05-14',
 '2026-05-15']

## 5. Select forecast date

In [5]:
# Choose one date from the printed list above.
# Default: tomorrow / second available date when available.
selected_date = forecast_dates[1] if len(forecast_dates) > 1 else forecast_dates[0]
hour_utc = 13

print("Selected date:", selected_date)
print("Hour UTC:", hour_utc)

Selected date: 2026-05-01
Hour UTC: 13


## 6. Helper functions

In [6]:
def norm(img, min_val, max_val):
    return (
        img.subtract(min_val)
        .divide(ee.Number(max_val).subtract(min_val))
        .clamp(0, 1)
    )

def mean_over_area(img, geom, scale=28000):
    result = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geom,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9,
    )
    return ee.Number(result.values().get(0))

## 7. Select forecast image closest to selected date at 13:00 UTC

In [7]:
target = ee.Date(f"{selected_date}T{hour_utc:02d}:00:00")

forecast_img = ee.Image(
    latest_run.map(
        lambda img: img.set(
            "timeDiff",
            ee.Number(img.get("forecast_time")).subtract(target.millis()).abs()
        )
    )
    .sort("timeDiff")
    .first()
)

print("Chosen forecast time:", ee.Date(ee.Number(forecast_img.get("forecast_time"))).format("YYYY-MM-dd HH:mm").getInfo(), "UTC")
print("Forecast hours:", forecast_img.get("forecast_hours").getInfo())
print("Available bands:")
forecast_img.bandNames().getInfo()

Chosen forecast time: 2026-05-01 12:00 UTC
Forecast hours: 24
Available bands:


['u_component_of_wind_100m_sfc',
 'v_component_of_wind_100m_sfc',
 'u_component_of_wind_10m_sfc',
 'v_component_of_wind_10m_sfc',
 'dewpoint_temperature_2m_sfc',
 'temperature_2m_sfc',
 'snow_albedo_sfc',
 'eastward_turbulent_surface_stress_sfc',
 'divergence_pl100',
 'divergence_pl1000',
 'divergence_pl150',
 'divergence_pl200',
 'divergence_pl250',
 'divergence_pl300',
 'divergence_pl400',
 'divergence_pl50',
 'divergence_pl500',
 'divergence_pl600',
 'divergence_pl700',
 'divergence_pl850',
 'divergence_pl925',
 'geopotential_height_pl100',
 'geopotential_height_pl1000',
 'geopotential_height_pl150',
 'geopotential_height_pl200',
 'geopotential_height_pl250',
 'geopotential_height_pl300',
 'geopotential_height_pl400',
 'geopotential_height_pl50',
 'geopotential_height_pl500',
 'geopotential_height_pl600',
 'geopotential_height_pl700',
 'geopotential_height_pl850',
 'geopotential_height_pl925',
 'land_sea_mask_sfc',
 'mean_sea_level_pressure_sfc',
 'most_unstable_convective_available

## 8. Extract forecast variables

In [8]:
# Main variables
t_c = forecast_img.select("temperature_2m_sfc").rename("air_temperature_C")
td_c = forecast_img.select("dewpoint_temperature_2m_sfc").rename("dewpoint_temperature_C")

u10 = forecast_img.select("u_component_of_wind_10m_sfc")
v10 = forecast_img.select("v_component_of_wind_10m_sfc")
wind = u10.pow(2).add(v10.pow(2)).sqrt().rename("wind_speed_ms")

rain_mm_h = (
    forecast_img.select("total_precipitation_rate_sfc")
    .multiply(3600 * 1000)
    .rename("precipitation_rate_mm_h")
)

solar_j = forecast_img.select("surface_solar_radiation_downwards_sfc").rename("solar_radiation_Jm2")

soil_temp_c = (
    forecast_img.select("soil_temperature_sol1")
    .subtract(273.15)
    .rename("soil_temperature_C")
)

soil_moisture = forecast_img.select("volumetric_soil_moisture_sol1").rename("soil_moisture_layer1")

# Derived air moisture
es = t_c.expression(
    "0.6108 * exp(17.27 * T / (T + 237.3))",
    {"T": t_c}
).rename("saturation_vapor_pressure_kPa")

ea = td_c.expression(
    "0.6108 * exp(17.27 * Td / (Td + 237.3))",
    {"Td": td_c}
).rename("actual_vapor_pressure_kPa")

rh = ea.divide(es).multiply(100).clamp(0, 100).rename("relative_humidity_percent")
vpd = es.subtract(ea).max(0).rename("vapor_pressure_deficit_kPa")

## 9. Average weather values over the protected area


In [9]:
scale = 28000

t_mean = mean_over_area(t_c, roisrp_geom, scale)
td_mean = mean_over_area(td_c, roisrp_geom, scale)
rh_mean = mean_over_area(rh, roisrp_geom, scale)
vpd_mean = mean_over_area(vpd, roisrp_geom, scale)
wind_mean = mean_over_area(wind, roisrp_geom, scale)
rain_mean = mean_over_area(rain_mm_h, roisrp_geom, scale)
solar_mean = mean_over_area(solar_j, roisrp_geom, scale)
soil_temp_mean = mean_over_area(soil_temp_c, roisrp_geom, scale)
soil_moisture_mean = mean_over_area(soil_moisture, roisrp_geom, scale)

meteo_values = {
    "Air temperature (C)": t_mean.getInfo(),
    "Dewpoint temperature (C)": td_mean.getInfo(),
    "Relative humidity (%)": rh_mean.getInfo(),
    "VPD (kPa)": vpd_mean.getInfo(),
    "Wind speed (m/s)": wind_mean.getInfo(),
    "Precipitation rate (mm/h)": rain_mean.getInfo(),
    "Solar radiation (J/m2)": solar_mean.getInfo(),
    "Soil temperature (C)": soil_temp_mean.getInfo(),
    "Soil moisture": soil_moisture_mean.getInfo(),
}

for k, v in meteo_values.items():
    print(f"{k}: {v:.2f}")

Air temperature (C): 14.88
Dewpoint temperature (C): 0.52
Relative humidity (%): 37.54
VPD (kPa): 1.06
Wind speed (m/s): 1.49
Precipitation rate (mm/h): 0.00
Solar radiation (J/m2): 21056800.00
Soil temperature (C): 17.61
Soil moisture: 0.11


## 10. Build dynamic modifier and final fire-risk raster

In [10]:
# Constant images from area-averaged values
t_const = ee.Image.constant(t_mean).clip(roi_geom)
rh_const = ee.Image.constant(rh_mean).clip(roi_geom)
vpd_const = ee.Image.constant(vpd_mean).clip(roi_geom)
wind_const = ee.Image.constant(wind_mean).clip(roi_geom)
rain_const = ee.Image.constant(rain_mean).clip(roi_geom)
solar_const = ee.Image.constant(solar_mean).clip(roi_geom)
soil_temp_const = ee.Image.constant(soil_temp_mean).clip(roi_geom)
soil_moisture_const = ee.Image.constant(soil_moisture_mean).clip(roi_geom)

# Normalized fire-weather risk terms
t_risk = norm(t_const, 15, 38)
vpd_risk = norm(vpd_const, 0.5, 3.5)
wind_risk = norm(wind_const, 1, 12)
solar_risk = norm(solar_const, 0, 3.0e7)
rain_dry_risk = ee.Image(1).subtract(norm(rain_const, 0.1, 2.0))
soil_dry_risk = ee.Image(1).subtract(norm(soil_moisture_const, 0.12, 0.38))
soil_temp_risk = norm(soil_temp_const, 10, 32)

# Weather index
weather_index = (
    t_risk.multiply(0.22)
    .add(vpd_risk.multiply(0.22))
    .add(wind_risk.multiply(0.22))
    .add(solar_risk.multiply(0.12))
    .add(rain_dry_risk.multiply(0.10))
    .add(soil_dry_risk.multiply(0.07))
    .add(soil_temp_risk.multiply(0.05))
    .rename("weather_index")
)

# Dynamic modifier
modifier = ee.Image(0.2).add(weather_index.multiply(1.6)).rename("dynamic_modifier")

# Suppression under wet conditions
modifier = modifier.where(rain_const.gt(2.0), 0.15)
modifier = modifier.where(rh_const.gt(95), 0.25)
modifier = modifier.where(rain_const.gt(0.5).And(rh_const.gt(85)), 0.35)

# Final fire risk
fire_risk = (
    stat4.multiply(modifier)
    .clamp(0, 100)
    .rename("short_term_forecast_fire_risk")
    .updateMask(stat4.gt(0))
)

## 11. Diagnostics

In [11]:
def area_mean(img, name):
    value = mean_over_area(img.rename("x"), roisrp_geom, scale).getInfo()
    print(f"{name}: {value:.2f}")

area_mean(weather_index, "Weather index")
area_mean(modifier, "Dynamic modifier")
area_mean(fire_risk, "Final fire risk")

Weather index: 0.32
Dynamic modifier: 0.72
Final fire risk: 12.17


## 12. Interactive map

In [12]:
risk_palette = [
    "#006400", "#228B22", "#7FBF3F",
    "#ADFF2F", "#FFFF00", "#FFD700",
    "#FFA500", "#FF7F00", "#FF4500",
    "#8B0000",
]

m = geemap.Map()
m.centerObject(roisrp, 10)
m.add_basemap("HYBRID")

m.addLayer(
    fire_risk.clip(roi_geom),
    {"min": 0, "max": 100, "palette": risk_palette},
    f"Short-term forecast fire risk {selected_date} {hour_utc}:00 UTC",
)

m.addLayer(
    ee.Image().paint(roisrp, 1, 2),
    {"palette": ["00FFFF"]},
    "Protected area boundary",
)

m.addLayer(
    weather_index.clip(roi_geom),
    {"min": 0, "max": 1, "palette": ["blue", "cyan", "yellow", "orange", "red"]},
    "Weather index",
    shown=False,
)

m.addLayer(
    modifier.clip(roi_geom),
    {"min": 0.15, "max": 1.8, "palette": ["blue", "cyan", "yellow", "orange", "red"]},
    "Dynamic modifier",
    shown=False,
)

m.addLayerControl()
m

Map(center=[44.89675093876204, 21.12246746431303], controls=(WidgetControl(options=['position', 'transparent_b…

## 13. Optional export

In [ ]:
# Uncomment to export the result as GeoTIFF.
# This may take some time depending on the region and scale.

# geemap.ee_export_image(
#     fire_risk.clip(roi_geom),
#     filename=f"outputs/short_term_forecast_fire_risk_{selected_date}_{hour_utc:02d}UTC.tif",
#     scale=100,
#     region=roi_geom,
#     file_per_band=False,
# )